# Assignment 3 — MM-IMDb M2

Unimodal image baseline, unimodal text baseline, and a simple fusion baseline.

- Image: ResNet-18, ImageNet weights, fine-tuned.
- Text: DistilBERT-base-uncased, fine-tuned.
- Simple fusion: frozen image and text encoders, cached features, concatenation head.
- Metrics: macro-F1, micro-F1, sample-wise F1, exact match.

[TODO] GMU, modality dropout, ablation, missing-modality tests, and error analysis remain for M3.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "assignment3").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "assignment3" / "src"))

import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

from utils.mmimdb import (
    build_label_matrix,
    build_splits,
    get_genres,
    load_mmimdb,
    make_image_loader,
    make_text_loader,
)
from models import (
    TEXT_MODEL_NAME,
    build_fusion_head,
    build_image_encoder,
    build_image_model,
    build_text_encoder,
    build_text_model,
)
from training_loop import (
    extract_image_features,
    extract_text_features,
    fit_fusion,
    fit_image,
    fit_text,
    metrics_from_probabilities,
    run_feature_epoch,
    run_image_epoch,
    run_text_epoch,
)

SEED = 42
IMAGE_EPOCHS = 10
TEXT_EPOCHS = 5
FUSION_EPOCHS = 30
IMAGE_BATCH = 256
TEXT_BATCH = 64
FUSION_BATCH = 256
WORKERS = 8
LIMIT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

RESULTS_DIR = ROOT / "assignment3" / "results"
CHECKPOINT_DIR = ROOT / "assignment3" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)

device: cuda


In [2]:
dataset = load_mmimdb()
genres = get_genres(dataset)
labels = build_label_matrix(dataset, genres)
splits = build_splits(dataset, genres)

if LIMIT:
    splits = {name: indices[:LIMIT] for name, indices in splits.items()}

print("genres:", len(genres))
print("splits:", {name: len(indices) for name, indices in splits.items()})
counts = labels.sum(axis=1)
print("labels per movie: mean", float(counts.mean()), "min", int(counts.min()), "max", int(counts.max()))


genres: 23
splits: {'train': 15558, 'dev': 2595, 'test': 7806}
labels per movie: mean 2.4846489429473877 min 1 max 10


In [3]:
image_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
image_model = build_image_model(len(genres)).to(DEVICE)
image_history, image_best_epoch, image_thresholds = fit_image(
    image_model,
    image_loaders,
    DEVICE,
    epochs=IMAGE_EPOCHS,
    lr=3e-4,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "image_resnet18.pt",
)

criterion = nn.BCEWithLogitsLoss()
_, image_test_probs, image_test_targets = run_image_epoch(
    image_model, image_loaders["test"], criterion, DEVICE, AMP
)
image_metrics = metrics_from_probabilities(
    image_test_probs, image_test_targets, image_thresholds
)
print({key: image_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/10 | train loss 0.3203 | val loss 0.2595 macro-F1 0.3134 | 12.1s


epoch 02/10 | train loss 0.2379 | val loss 0.2548 macro-F1 0.3356 | 9.7s


epoch 03/10 | train loss 0.2150 | val loss 0.2617 macro-F1 0.3243 | 9.6s


epoch 04/10 | train loss 0.1882 | val loss 0.2823 macro-F1 0.3248 | 9.7s


epoch 05/10 | train loss 0.1596 | val loss 0.2851 macro-F1 0.3182 | 10.3s


epoch 06/10 | train loss 0.1325 | val loss 0.3098 macro-F1 0.3205 | 9.8s


epoch 07/10 | train loss 0.1083 | val loss 0.3224 macro-F1 0.3060 | 9.5s


epoch 08/10 | train loss 0.0890 | val loss 0.3459 macro-F1 0.2960 | 9.5s


epoch 09/10 | train loss 0.0718 | val loss 0.3357 macro-F1 0.3057 | 9.5s


epoch 10/10 | train loss 0.0599 | val loss 0.3513 macro-F1 0.3078 | 9.4s


{'macro_f1': 0.330435024580552, 'micro_f1': 0.42400275103163687, 'samples_f1': 0.4206431621016556, 'exact_match': 0.022162439149372277}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, name == "train", WORKERS)
    for name, indices in splits.items()
}
text_model = build_text_model(len(genres)).to(DEVICE)
text_history, text_best_epoch, text_thresholds = fit_text(
    text_model,
    text_loaders,
    DEVICE,
    epochs=TEXT_EPOCHS,
    lr=2e-5,
    weight_decay=1e-4,
    amp=AMP,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "text_distilbert.pt",
)

_, text_test_probs, text_test_targets = run_text_epoch(
    text_model, text_loaders["test"], criterion, DEVICE, AMP
)
text_metrics = metrics_from_probabilities(
    text_test_probs, text_test_targets, text_thresholds
)
print({key: text_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch 01/5 | train loss 0.3329 | val loss 0.2438 macro-F1 0.3824 | 14.2s


epoch 02/5 | train loss 0.2168 | val loss 0.2085 macro-F1 0.5089 | 13.6s


epoch 03/5 | train loss 0.1844 | val loss 0.1892 macro-F1 0.5563 | 13.6s


epoch 04/5 | train loss 0.1633 | val loss 0.1836 macro-F1 0.5737 | 13.6s


epoch 05/5 | train loss 0.1453 | val loss 0.1868 macro-F1 0.5844 | 13.7s


{'macro_f1': 0.5714667603353641, 'micro_f1': 0.6415780035863717, 'samples_f1': 0.6454644637965161, 'exact_match': 0.1614142966948501}


In [5]:
image_encoder = build_image_encoder().to(DEVICE)
text_encoder = build_text_encoder().to(DEVICE)

image_feature_loaders = {
    name: make_image_loader(dataset, indices, labels, IMAGE_BATCH, False, WORKERS)
    for name, indices in splits.items()
}
text_feature_loaders = {
    name: make_text_loader(dataset, indices, labels, tokenizer, TEXT_BATCH, False, WORKERS)
    for name, indices in splits.items()
}

fusion_features = {}
fusion_labels = {}
for name in splits:
    image_features, image_targets = extract_image_features(
        image_encoder, image_feature_loaders[name], DEVICE, AMP
    )
    text_features, text_targets = extract_text_features(
        text_encoder, text_feature_loaders[name], DEVICE, AMP
    )
    assert torch.equal(image_targets, text_targets)
    fusion_features[name] = torch.cat([image_features, text_features], dim=1)
    fusion_labels[name] = image_targets

fusion_head = build_fusion_head(
    input_dim=fusion_features["train"].shape[1],
    hidden_dim=512,
    num_labels=len(genres),
).to(DEVICE)
fusion_history, fusion_best_epoch, fusion_thresholds = fit_fusion(
    fusion_head,
    fusion_features["train"],
    fusion_labels["train"],
    fusion_features["dev"],
    fusion_labels["dev"],
    DEVICE,
    epochs=FUSION_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=FUSION_BATCH,
    seed=SEED,
    checkpoint_path=CHECKPOINT_DIR / "fusion_mlp.pt",
)

dev_loader = DataLoader(
    TensorDataset(fusion_features["dev"], fusion_labels["dev"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(fusion_features["test"], fusion_labels["test"]),
    batch_size=FUSION_BATCH,
    shuffle=False,
)
_, fusion_test_probs, fusion_test_targets = run_feature_epoch(
    fusion_head, test_loader, criterion, DEVICE
)
fusion_metrics = metrics_from_probabilities(
    fusion_test_probs, fusion_test_targets, fusion_thresholds
)
print({key: fusion_metrics[key] for key in ("macro_f1", "micro_f1", "samples_f1", "exact_match")})

epoch 01/30 | train loss 0.2811 | val macro-F1 0.4215


epoch 05/30 | train loss 0.1845 | val macro-F1 0.5687


epoch 10/30 | train loss 0.1723 | val macro-F1 0.5884


epoch 15/30 | train loss 0.1661 | val macro-F1 0.5902


epoch 20/30 | train loss 0.1606 | val macro-F1 0.5931


epoch 25/30 | train loss 0.1566 | val macro-F1 0.5904


epoch 30/30 | train loss 0.1537 | val macro-F1 0.5949
{'macro_f1': 0.5814628008729688, 'micro_f1': 0.6419317412742918, 'samples_f1': 0.6385478379905745, 'exact_match': 0.14693825262618498}


In [6]:
rows = []
for name, model, metrics, history, best_epoch, thresholds in [
    ("Image-only ResNet-18", image_model, image_metrics, image_history, image_best_epoch, image_thresholds),
    ("Text-only DistilBERT", text_model, text_metrics, text_history, text_best_epoch, text_thresholds),
    ("Simple fusion", fusion_head, fusion_metrics, fusion_history, fusion_best_epoch, fusion_thresholds),
]:
    rows.append(
        {
            "model": name,
            "params": sum(p.numel() for p in model.parameters()),
            "best_epoch": int(best_epoch),
            "train_time_s": float(sum(history["epoch_time"])),
            "test_macro_f1": metrics["macro_f1"],
            "test_micro_f1": metrics["micro_f1"],
            "test_samples_f1": metrics["samples_f1"],
            "test_exact_match": metrics["exact_match"],
        }
    )

results_df = pd.DataFrame(rows)
results_df

,model,params,best_epoch,train_time_s,test_macro_f1,test_micro_f1,test_samples_f1,test_exact_match
0,Image-only ResNet-18,11188311,2,99.105152,0.330435,0.424003,0.420643,0.022162
1,Text-only DistilBERT,66971159,5,68.689335,0.571467,0.641578,0.645464,0.161414
2,Simple fusion,667671,24,25.468700,0.581463,0.641932,0.638548,0.146938


In [7]:
config = {
    "dataset": "MM-IMDb",
    "task": "23-label movie genre classification",
    "seed": SEED,
    "image_epochs": IMAGE_EPOCHS,
    "text_epochs": TEXT_EPOCHS,
    "fusion_epochs": FUSION_EPOCHS,
    "image_batch": IMAGE_BATCH,
    "text_batch": TEXT_BATCH,
    "fusion_batch": FUSION_BATCH,
    "limit": LIMIT,
    "amp": AMP,
    "device": str(DEVICE),
}
payload = {
    "config": config,
    "genres": genres,
    "splits": {name: len(indices) for name, indices in splits.items()},
    "results": rows,
    "history": {
        "image": image_history,
        "text": text_history,
        "fusion": fusion_history,
    },
    "thresholds": {
        "image": image_thresholds.tolist(),
        "text": text_thresholds.tolist(),
        "fusion": fusion_thresholds.tolist(),
    },
    "per_class_f1": {
        "image": image_metrics["per_class_f1"],
        "text": text_metrics["per_class_f1"],
        "fusion": fusion_metrics["per_class_f1"],
    },
}
with open(RESULTS_DIR / "m2_results.json", "w") as handle:
    json.dump(payload, handle, indent=2)
results_df.to_csv(RESULTS_DIR / "m2_results.csv", index=False)
print("saved", RESULTS_DIR / "m2_results.json")

saved /raid/home/an.nv/main/deep-learning/CO3133-DeepLearning/assignment3/results/m2_results.json
